# 📜 Legal DMS — Production-Grade RAG Pipeline (Colab Edition)

This notebook replicates the **doc-search** retrieval system from the DMS Knowledge Base project, enhanced with production-grade retrieval architecture concepts.

### Pipeline Architecture
```
                         QUERY
                           │
                  Query Understanding
                   (Intent + Entities)
                           │
                  Retrieval Planner
                   (Adaptive Channels)
                           │
         ┌─────────────────┼─────────────────┐
         ▼                 ▼                 ▼
    BM25 Keyword      Vector Cosine     Metadata Filter
         │                 │                 │
         └─────────────────┼─────────────────┘
                           ▼
                  Candidate Pool + Dedup
                    (with Provenance)
                           │
                     RRF Fusion
                           │
                   Cross-Encoder Rerank
                           │
                    Context Builder
                           │
                     LLM Answer
                    + Citations
```

### Production Concepts Implemented
- **Unified Retriever Protocol** — every channel returns `Candidate` with provenance
- **Retrieval Planner** — query understanding drives which channels run
- **Candidate Provenance** — every result tracks which channels found it and individual scores
- **3 Retrieval Channels** — BM25 keyword, vector cosine, metadata filter (all parallel)
- **Retrieval Debugger** — Gradio tab shows full channel breakdown per query
- **Folder-based ingestion** — point to any PDF folder (local or Google Drive)
- **Continuous Q&A** — Gradio UI with public URL stays active for interactive use

## 1. Install Dependencies

In [ ]:
!pip install -q sentence-transformers chromadb rank-bm25 pypdf gradio groq google-genai openai

## 2. Configuration & API Keys

Set your API keys below. The system tries **Groq → Gemini → extractive fallback** in order.

You can set keys via:
1. Colab Secrets (key icon in left sidebar) — recommended
2. Environment variables
3. Direct assignment below

In [ ]:
import os
from dataclasses import dataclass, field

# ── Attempt to load from Colab Secrets ────────────────────────────────────
try:
    from google.colab import userdata
    _GROQ_KEY = userdata.get('GROQ_API_KEY', '')
    _GEMINI_KEY = userdata.get('GEMINI_API_KEY', '')
except Exception:
    _GROQ_KEY = ''
    _GEMINI_KEY = ''


@dataclass
class Config:
    # LLM providers (tried in order: Groq -> Gemini -> extractive)
    groq_api_key: str = os.environ.get('GROQ_API_KEY', _GROQ_KEY)
    groq_model: str = 'llama-3.3-70b-versatile'

    gemini_api_key: str = os.environ.get('GEMINI_API_KEY', _GEMINI_KEY)
    gemini_model: str = 'gemini-2.0-flash'

    # Retrieval
    embedding_model: str = 'sentence-transformers/all-MiniLM-L6-v2'
    rerank_model: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
    rerank_ce_weight: float = 0.55
    retrieve_k: int = 8
    retrieve_limit: int = 40

    # Chunking
    chunk_max_chars: int = 1200
    chunk_overlap: int = 150


config = Config()
print('✅ Config initialized')
print(f'   Groq key:   {"✓ set" if config.groq_api_key else "✗ NOT SET"}')
print(f'   Gemini key: {"✓ set" if config.gemini_api_key else "✗ NOT SET"}')

## 3. Core Data Models — Unified Candidate Schema

Every retrieval channel returns the same `Candidate` dataclass with **provenance tracking**.
This is Phase 0 of the production roadmap: establish the contracts first.

```
Candidate(
    chunk_id, document_id, matter_id,
    channel, raw_score,
    metadata, provenance
)
```

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable
from enum import Enum, auto


# ── Query Understanding ──────────────────────────────────────────────────────

class QueryIntent(Enum):
    """Detected query intent drives which retrieval channels activate."""
    EXACT_LOOKUP = auto()      # "Find document X" → Metadata + BM25
    NORMAL_RESEARCH = auto()   # "What are the force majeure claims?" → All channels
    SIMILAR_MATTER = auto()    # "Similar cases to X" → Vector + Matter
    CLIENT_HISTORY = auto()    # "Cases for Avaada" → Metadata + Matter
    GENERAL = auto()           # Fallback → BM25 + Vector


@dataclass
class ParsedQuery:
    """Result of query understanding."""
    raw_query: str
    intent: QueryIntent = QueryIntent.GENERAL
    entities: dict = field(default_factory=dict)  # client, matter_id, doc_type, etc.
    filters: dict = field(default_factory=dict)   # metadata filters to apply


# ── Unified Candidate Schema ─────────────────────────────────────────────────

@dataclass
class Candidate:
    """Unified candidate returned by every retrieval channel.

    Provenance tracking means fusion doesn't care whether the candidate
    came from BM25, vector search, metadata filter, or graph traversal.
    """
    chunk_id: str
    file_id: str
    filename: str
    page_number: int
    text: str

    # Channel info
    channel: str = ''
    raw_score: float = 0.0

    # DMS metadata
    matter_id: str | None = None
    document_type: str | None = None
    tags: list[str] = field(default_factory=list)
    forum: str | None = None
    case_number: str | None = None
    client_name: str | None = None

    # Scores filled at various pipeline stages
    fused_score: float = 0.0
    ce_score: float = 0.0
    rerank_score: float = 0.0

    # Provenance: which channels found this candidate and their scores
    provenance: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            'chunk_id': self.chunk_id,
            'file_id': self.file_id,
            'filename': self.filename,
            'page_number': self.page_number,
            'text': self.text,
            'channel': self.channel,
            'raw_score': self.raw_score,
            'matter_id': self.matter_id,
            'document_type': self.document_type,
            'tags': self.tags,
            'forum': self.forum,
            'case_number': self.case_number,
            'client_name': self.client_name,
            'fused_score': self.fused_score,
            'ce_score': self.ce_score,
            'rerank_score': self.rerank_score,
            'provenance': self.provenance,
        }


# ── Unified Retriever Protocol ───────────────────────────────────────────────

@runtime_checkable
class Retriever(Protocol):
    """Every retrieval channel implements this interface.

    This lets us swap BM25/Vector/Metadata/Graph implementations
    without touching the retrieval engine.
    """
    @property
    def channel_name(self) -> str: ...

    def retrieve(self, query: ParsedQuery, limit: int) -> list[Candidate]: ...


print('✅ Data models ready (Candidate, ParsedQuery, Retriever Protocol)')

## 4. Text Chunking

Legal-document-aware chunking: 1200 char chunks with 150 char overlap, preferring paragraph/newline boundaries.

In [ ]:
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 150) -> list[str]:
    """Split text into chunks, preferring paragraph boundaries."""
    text = (text or '').strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    parts: list[str] = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        if end < len(text):
            cut = text.rfind('\n\n', start, end)
            if cut > start + max_chars // 2:
                end = cut
            else:
                cut = text.rfind('\n', start, end)
                if cut > start + max_chars // 2:
                    end = cut
        piece = text[start:end].strip()
        if piece:
            parts.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return parts


print('✅ Chunking function ready')

## 5. Metadata Extraction

Regex-based extraction of DMS metadata from Indian legal documents.

In [ ]:
import re

_DOCTYPE_PATTERNS = [
    ('Affidavit', re.compile(r'\baffidavit\b', re.I)),
    ('Written Submission', re.compile(r'\bwritten\s+submission\b', re.I)),
    ('Brief Note', re.compile(r'\bbrief\s+note\b', re.I)),
    ('Rejoinder', re.compile(r'\brejoinder\b', re.I)),
    ('Reply', re.compile(r'\breply\b', re.I)),
    ('Writ Petition', re.compile(r'\bwrit\s+petition\b', re.I)),
    ('Petition', re.compile(r'\bpetition\b', re.I)),
    ('Research Memo', re.compile(r'\bresearch\s+(memo|note)\b', re.I)),
    ('Final Reply', re.compile(r'\bfinal\s+reply\b', re.I)),
    ('Impleadment Application', re.compile(r'\bimpleadment\b', re.I)),
]

_FORUM_PATTERNS = [
    ('CERC', re.compile(r'\bCERC\b')),
    ('APTEL', re.compile(r'\bAPTEL\b')),
    ('MERC', re.compile(r'\bMERC\b')),
    ('MSEDCL', re.compile(r'\bMSEDCL\b')),
    ('Supreme Court', re.compile(r'\bSupreme\s+Court\b', re.I)),
    ('High Court', re.compile(r'\bHigh\s+Court\b', re.I)),
    ('NCLT', re.compile(r'\bNCLT\b')),
    ('NCLAT', re.compile(r'\bNCLAT\b')),
    ('SEBI', re.compile(r'\bSEBI\b')),
]

_CASE_NUMBER_RES = [
    re.compile(r'P\.?\s*N\.?\s*(\d+\s+of\s+\d{4})', re.I),
    re.compile(r'Petition\s+No\.?\s*(\d+\s+of\s+\d{4})', re.I),
    re.compile(r'(?:CA|Civil\s+Appeal)\s+(\d+\s+of\s+\d{4})', re.I),
    re.compile(r'(?:MP|Misc\.?\s*Petition)\s+(\d+\s+of\s+\d{4})', re.I),
    re.compile(r'(?:APL|Appeal)\s*\.?\s*(\d+\s+of\s+\d{4})', re.I),
    re.compile(r'(\d+-MP-\d{4})', re.I),
]

_MATTER_ID_RE = re.compile(r'\bMWSP[_\-]PROJ\d+\b', re.I)
_MATTER_ID_ALT = re.compile(r'\bMTR-\d{4}-\d+\b', re.I)

_CLIENT_PATTERNS = [
    ('Avaada Energy Pvt. Ltd.', re.compile(r'\bAvaada\b', re.I)),
    ('MSEDCL', re.compile(r'\bMSEDCL\b')),
    ('Vector Green Energy Pvt. Ltd.', re.compile(r'\bVector\s+Green\b', re.I)),
    ('AEPL', re.compile(r'\bAEPL\b')),
    ('CTUIL', re.compile(r'\bCTUIL\b')),
]

_TAG_PATTERNS = [
    ('Force Majeure', re.compile(r'\bforce\s+majeure\b', re.I)),
    ('Wind Power', re.compile(r'\bwind\s+power\b|\bwind\s+energy\b', re.I)),
    ('Solar Power', re.compile(r'\bsolar\s+power\b|\bsolar\s+energy\b', re.I)),
    ('Regulatory', re.compile(r'\bregulatory\b|\bregulation\b', re.I)),
    ('Tariff', re.compile(r'\btariff\b', re.I)),
    ('PPA', re.compile(r'\bPPA\b|\bpower\s+purchase\s+agreement\b', re.I)),
    ('Arbitration', re.compile(r'\barbitration\b', re.I)),
    ('Commissioning', re.compile(r'\bcommission(?:ing)?\b', re.I)),
    ('Change in Law', re.compile(r'\bchange\s+in\s+law\b', re.I)),
    ('Electricity', re.compile(r'\belectricity\b', re.I)),
    ('Connectivity', re.compile(r'\bconnectivity\b', re.I)),
    ('Transmission', re.compile(r'\btransmission\b', re.I)),
    ('Flooding', re.compile(r'\bflood(?:ing|s)?\b', re.I)),
    ('Rainfall', re.compile(r'\brainfall\b|\bheavy\s+rain\b', re.I)),
    ('Gujarat', re.compile(r'\bGujarat\b', re.I)),
    ('Maharashtra', re.compile(r'\bMaharashtra\b', re.I)),
    ('Delay', re.compile(r'\bdelay(?:s|ed)?\b', re.I)),
    ('Indemnity', re.compile(r'\bindemnity\b|\bindemnification\b', re.I)),
]


def extract_metadata(filename: str, full_text: str) -> dict:
    """Extract all DMS metadata from a PDF filename and text."""
    combined = f'{filename}\n{full_text[:2000]}'

    doc_type = 'Legal Document'
    for dtype, pattern in _DOCTYPE_PATTERNS:
        if pattern.search(combined):
            doc_type = dtype
            break

    forum = None
    for f, pattern in _FORUM_PATTERNS:
        if pattern.search(full_text[:5000]):
            forum = f
            break

    case_number = None
    for regex in _CASE_NUMBER_RES:
        m = regex.search(full_text[:5000])
        if m:
            case_number = m.group(0).strip()
            break

    matter_id = None
    m = _MATTER_ID_RE.search(full_text[:10000])
    if m:
        matter_id = m.group(0)
    else:
        m = _MATTER_ID_ALT.search(full_text[:10000])
        if m:
            matter_id = m.group(0)

    client_name = None
    for client, pattern in _CLIENT_PATTERNS:
        if pattern.search(full_text[:5000]):
            client_name = client
            break

    tag_combined = f'{filename}\n{full_text[:8000]}'
    tags = [tag for tag, pattern in _TAG_PATTERNS if pattern.search(tag_combined)]

    return {
        'document_type': doc_type,
        'forum': forum,
        'case_number': case_number,
        'matter_id': matter_id,
        'client_name': client_name,
        'tags': tags,
    }


print('✅ Metadata extractor ready')

## 6. Load Models (Embedder + Cross-Encoder Reranker)

In [ ]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

print('Loading embedding model...')
t0 = time.time()
embedder = SentenceTransformer(config.embedding_model)
print(f'  ✅ Embedder loaded in {time.time()-t0:.1f}s')

print('Loading cross-encoder reranker...')
t0 = time.time()
reranker = CrossEncoder(config.rerank_model)
print(f'  ✅ Reranker loaded in {time.time()-t0:.1f}s')


def encode_texts(texts: list[str]) -> list[list[float]]:
    """Encode texts with MiniLM, normalized."""
    if not texts:
        return []
    vectors = embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return [row.tolist() for row in np.asarray(vectors, dtype=np.float32)]

## 7. Document Ingestion + In-Memory Stores

Point to a folder of PDFs. The system:
1. Extracts text from each PDF page (pypdf)
2. Chunks with legal-aware splitting (1200 chars, 150 overlap)
3. Extracts DMS metadata (document type, forum, case number, tags)
4. Embeds with MiniLM-L6-v2 → ChromaDB (vector store)
5. Builds BM25 index (keyword search)
6. Builds metadata index (metadata retrieval channel)

In [ ]:
import hashlib
from pathlib import Path
from pypdf import PdfReader
import chromadb
from rank_bm25 import BM25Okapi
from collections import defaultdict


# ── Global stores (Storage abstraction — today in-memory, later dedicated DBs) ─
chroma_client = chromadb.Client()
collection = None            # ChromaDB collection
bm25_index = None            # BM25Okapi index
all_chunks: list[dict] = []  # Master chunk list
chunk_id_map: dict = {}      # chunk_id -> chunk dict

# Metadata indices for the metadata retrieval channel
meta_by_doc_type: dict[str, list[str]] = defaultdict(list)  # doc_type -> [chunk_ids]
meta_by_forum: dict[str, list[str]] = defaultdict(list)     # forum -> [chunk_ids]
meta_by_client: dict[str, list[str]] = defaultdict(list)    # client -> [chunk_ids]
meta_by_matter: dict[str, list[str]] = defaultdict(list)    # matter_id -> [chunk_ids]
meta_by_tag: dict[str, list[str]] = defaultdict(list)       # tag -> [chunk_ids]


def _file_id(path: Path) -> str:
    return hashlib.sha1(path.name.encode()).hexdigest()[:16]


def _chunk_id(fid: str, idx: int) -> str:
    return f'{fid}_{idx:05d}'


def extract_pages(pdf_path: Path) -> list[tuple[int, str]]:
    """Returns list of (1-indexed page_number, text)."""
    try:
        reader = PdfReader(str(pdf_path))
        pages = []
        for i, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or '').strip()
            if text:
                pages.append((i, text))
        return pages
    except Exception as e:
        print(f'  ⚠️ Error reading {pdf_path.name}: {e}')
        return []


def ingest_folder(folder_path: str) -> str:
    """Ingest all PDFs from a folder. Returns status message."""
    global collection, bm25_index, all_chunks, chunk_id_map
    global meta_by_doc_type, meta_by_forum, meta_by_client, meta_by_matter, meta_by_tag

    folder = Path(folder_path)
    if not folder.exists():
        return f'❌ Folder not found: {folder_path}'
    if not folder.is_dir():
        return f'❌ Not a directory: {folder_path}'

    pdfs = sorted(folder.glob('*.pdf'))
    if not pdfs:
        return f'❌ No PDF files found in: {folder_path}'

    # Reset all stores
    try:
        chroma_client.delete_collection('legal_docs')
    except Exception:
        pass
    collection = chroma_client.create_collection(
        name='legal_docs',
        metadata={'hnsw:space': 'cosine'},
    )
    all_chunks = []
    chunk_id_map = {}
    meta_by_doc_type = defaultdict(list)
    meta_by_forum = defaultdict(list)
    meta_by_client = defaultdict(list)
    meta_by_matter = defaultdict(list)
    meta_by_tag = defaultdict(list)

    total_chunks = 0
    lines = [f'📂 Found {len(pdfs)} PDFs in {folder_path}', '']

    for pdf_path in pdfs:
        fid = _file_id(pdf_path)
        filename = pdf_path.name
        print(f'Processing: {filename}')

        pages = extract_pages(pdf_path)
        if not pages:
            lines.append(f'  ⏭️ SKIP (no text): {filename}')
            continue

        full_text = '\n\n'.join(text for _, text in pages)
        meta = extract_metadata(filename, full_text)

        # Chunk
        chunks = []
        idx = 0
        for page_num, page_text in pages:
            for piece in chunk_text(page_text, config.chunk_max_chars, config.chunk_overlap):
                cid = _chunk_id(fid, idx)
                chunks.append({
                    'chunk_id': cid,
                    'file_id': fid,
                    'filename': filename,
                    'page_number': page_num,
                    'chunk_index': idx,
                    'text': piece,
                    'matter_id': meta['matter_id'],
                    'document_type': meta['document_type'],
                    'tags': meta['tags'],
                    'forum': meta['forum'],
                    'case_number': meta['case_number'],
                    'client_name': meta['client_name'],
                })
                idx += 1

        if not chunks:
            lines.append(f'  ⏭️ SKIP (no chunks): {filename}')
            continue

        # Embed
        texts = [c['text'] for c in chunks]
        t0 = time.time()
        embeddings = encode_texts(texts)
        embed_time = time.time() - t0

        # Add to ChromaDB
        collection.add(
            ids=[c['chunk_id'] for c in chunks],
            embeddings=embeddings,
            documents=texts,
            metadatas=[
                {
                    'filename': c['filename'],
                    'page_number': c['page_number'],
                    'document_type': c['document_type'] or '',
                    'matter_id': c['matter_id'] or '',
                    'tags': ','.join(c['tags']) if c['tags'] else '',
                    'forum': c['forum'] or '',
                    'case_number': c['case_number'] or '',
                    'client_name': c['client_name'] or '',
                }
                for c in chunks
            ],
        )

        # Add to master list + metadata indices
        for c in chunks:
            all_chunks.append(c)
            chunk_id_map[c['chunk_id']] = c

            # Build metadata indices
            if c['document_type']:
                meta_by_doc_type[c['document_type'].lower()].append(c['chunk_id'])
            if c['forum']:
                meta_by_forum[c['forum'].lower()].append(c['chunk_id'])
            if c['client_name']:
                meta_by_client[c['client_name'].lower()].append(c['chunk_id'])
            if c['matter_id']:
                meta_by_matter[c['matter_id'].lower()].append(c['chunk_id'])
            for tag in (c['tags'] or []):
                meta_by_tag[tag.lower()].append(c['chunk_id'])

        total_chunks += len(chunks)
        lines.append(
            f'  ✅ {filename}: {len(chunks)} chunks, '
            f'type={meta["document_type"]}, '
            f'embed={embed_time:.1f}s'
        )

    # Build BM25 index
    if all_chunks:
        tokenized = [c['text'].lower().split() for c in all_chunks]
        bm25_index = BM25Okapi(tokenized)

    lines.append('')
    lines.append(f'🎉 Done! {total_chunks} chunks across {len(pdfs)} files.')
    lines.append(f'   Vector store: ChromaDB ({collection.count()} vectors)')
    lines.append(f'   Keyword index: BM25 ({len(all_chunks)} documents)')
    lines.append(f'   Metadata indices: {len(meta_by_doc_type)} doc types, '
                 f'{len(meta_by_forum)} forums, {len(meta_by_client)} clients, '
                 f'{len(meta_by_matter)} matters, {len(meta_by_tag)} tags')

    result = '\n'.join(lines)
    print(result)
    return result


print('✅ Ingestion pipeline ready')

## 8. Query Understanding + Retrieval Planner

**Phase 1 from the roadmap**: query understanding detects intent and entities,
then the retrieval planner decides which channels to activate.

```
EXACT_LOOKUP     → Metadata + BM25
NORMAL_RESEARCH  → BM25 + Vector + Metadata
SIMILAR_MATTER   → Vector + Metadata
CLIENT_HISTORY   → Metadata + BM25
GENERAL          → BM25 + Vector
```

This gives us **query-adaptive compute** — don't run expensive channels unnecessarily.

In [ ]:
# ── Entity extraction patterns ───────────────────────────────────────────────
_CLIENT_NAMES = {
    'avaada': 'Avaada Energy Pvt. Ltd.',
    'msedcl': 'MSEDCL',
    'vector green': 'Vector Green Energy Pvt. Ltd.',
    'aepl': 'AEPL',
    'ctuil': 'CTUIL',
}

_FORUM_NAMES = {
    'cerc': 'CERC', 'aptel': 'APTEL', 'merc': 'MERC',
    'supreme court': 'Supreme Court', 'high court': 'High Court',
    'nclt': 'NCLT', 'nclat': 'NCLAT', 'sebi': 'SEBI',
}

_DOC_TYPES = {
    'affidavit': 'Affidavit', 'petition': 'Petition',
    'written submission': 'Written Submission', 'rejoinder': 'Rejoinder',
    'reply': 'Reply', 'brief note': 'Brief Note',
    'writ petition': 'Writ Petition', 'research memo': 'Research Memo',
}


def understand_query(raw_query: str) -> ParsedQuery:
    """Lightweight query understanding: detect intent and extract entities."""
    q_lower = raw_query.lower().strip()
    entities = {}
    filters = {}

    # ── Extract entities ──────────────────────────────────────────────────
    for key, name in _CLIENT_NAMES.items():
        if key in q_lower:
            entities['client'] = name
            filters['client_name'] = name
            break

    for key, name in _FORUM_NAMES.items():
        if key in q_lower:
            entities['forum'] = name
            filters['forum'] = name
            break

    for key, name in _DOC_TYPES.items():
        if key in q_lower:
            entities['document_type'] = name
            filters['document_type'] = name
            break

    # Matter ID
    m = _MATTER_ID_RE.search(raw_query) or _MATTER_ID_ALT.search(raw_query)
    if m:
        entities['matter_id'] = m.group(0)
        filters['matter_id'] = m.group(0)

    # ── Detect intent ─────────────────────────────────────────────────────
    intent = QueryIntent.GENERAL

    # Exact lookup: mentions a specific document/case number
    for regex in _CASE_NUMBER_RES:
        if regex.search(raw_query):
            intent = QueryIntent.EXACT_LOOKUP
            entities['case_number'] = regex.search(raw_query).group(0)
            break

    # Client history
    if intent == QueryIntent.GENERAL and 'client' in entities:
        client_patterns = ['cases for', 'matters for', 'history of', 'documents for',
                           'work for', 'handled for', 'all.*for']
        for p in client_patterns:
            if re.search(p, q_lower):
                intent = QueryIntent.CLIENT_HISTORY
                break

    # Similar matter
    if intent == QueryIntent.GENERAL:
        similar_patterns = ['similar', 'like', 'comparable', 'related to', 'analogous']
        for p in similar_patterns:
            if p in q_lower:
                intent = QueryIntent.SIMILAR_MATTER
                break

    # Normal research: has entities or is a substantive question
    if intent == QueryIntent.GENERAL and (entities or len(q_lower.split()) > 5):
        intent = QueryIntent.NORMAL_RESEARCH

    return ParsedQuery(
        raw_query=raw_query,
        intent=intent,
        entities=entities,
        filters=filters,
    )


# ── Retrieval Planner ─────────────────────────────────────────────────────────

# Map intent -> which channels to activate
CHANNEL_PLAN: dict[QueryIntent, list[str]] = {
    QueryIntent.EXACT_LOOKUP:    ['metadata', 'bm25'],
    QueryIntent.NORMAL_RESEARCH: ['bm25', 'vector', 'metadata'],
    QueryIntent.SIMILAR_MATTER:  ['vector', 'metadata'],
    QueryIntent.CLIENT_HISTORY:  ['metadata', 'bm25'],
    QueryIntent.GENERAL:         ['bm25', 'vector'],
}


def plan_retrieval(pq: ParsedQuery) -> list[str]:
    """Returns list of channel names to activate based on query intent."""
    return CHANNEL_PLAN.get(pq.intent, ['bm25', 'vector'])


# Quick test
_test = understand_query('What force majeure claims were filed in CERC petitions?')
print(f'✅ Query understanding ready')
print(f'   Test: "{_test.raw_query}"')
print(f'   Intent: {_test.intent.name}')
print(f'   Entities: {_test.entities}')
print(f'   Plan: {plan_retrieval(_test)}')

## 9. Retrieval Channels (BM25 + Vector + Metadata)

Each channel implements the same interface and returns `Candidate` objects with provenance.

```
BM25Retriever      → keyword search via BM25Okapi
VectorRetriever    → cosine similarity via ChromaDB
MetadataRetriever  → exact match on doc_type, forum, client, matter_id, tags
```

In [ ]:
def _chunk_to_candidate(chunk: dict, channel: str, score: float) -> Candidate:
    """Convert a raw chunk dict to a Candidate with channel provenance."""
    return Candidate(
        chunk_id=chunk['chunk_id'],
        file_id=chunk['file_id'],
        filename=chunk['filename'],
        page_number=chunk['page_number'],
        text=chunk['text'],
        channel=channel,
        raw_score=score,
        matter_id=chunk.get('matter_id'),
        document_type=chunk.get('document_type'),
        tags=chunk.get('tags', []),
        forum=chunk.get('forum'),
        case_number=chunk.get('case_number'),
        client_name=chunk.get('client_name'),
        provenance={channel: score},
    )


# ── Channel: BM25 Keyword Search ─────────────────────────────────────────────

class BM25Retriever:
    channel_name = 'bm25'

    def retrieve(self, query: ParsedQuery, limit: int = 40) -> list[Candidate]:
        if not bm25_index or not all_chunks:
            return []
        tokenized = query.raw_query.lower().split()
        scores = bm25_index.get_scores(tokenized)
        top_indices = np.argsort(scores)[::-1][:limit]
        return [
            _chunk_to_candidate(all_chunks[i], 'bm25', float(scores[i]))
            for i in top_indices if scores[i] > 0
        ]


# ── Channel: Vector Cosine Search ────────────────────────────────────────────

class VectorRetriever:
    channel_name = 'vector'

    def retrieve(self, query: ParsedQuery, limit: int = 40) -> list[Candidate]:
        if collection is None or collection.count() == 0:
            return []
        query_vec = encode_texts([query.raw_query])[0]
        results = collection.query(
            query_embeddings=[query_vec],
            n_results=min(limit, collection.count()),
            include=['documents', 'metadatas', 'distances'],
        )
        hits = []
        for i, cid in enumerate(results['ids'][0]):
            chunk = chunk_id_map.get(cid)
            if chunk:
                score = 1.0 - results['distances'][0][i]
                hits.append(_chunk_to_candidate(chunk, 'vector', score))
        return hits


# ── Channel: Metadata Filter Search ──────────────────────────────────────────

class MetadataRetriever:
    """Retrieves chunks matching detected metadata entities in the query.

    This is a separate retrieval channel — not a post-filter.
    It returns candidates that match by document_type, forum, client,
    matter_id, or tags detected during query understanding.
    """
    channel_name = 'metadata'

    def retrieve(self, query: ParsedQuery, limit: int = 40) -> list[Candidate]:
        if not query.filters:
            return []

        # Collect matching chunk_ids from each filter dimension
        candidate_ids: dict[str, int] = defaultdict(int)  # chunk_id -> match_count

        filters = query.filters
        if 'document_type' in filters:
            for cid in meta_by_doc_type.get(filters['document_type'].lower(), []):
                candidate_ids[cid] += 1
        if 'forum' in filters:
            for cid in meta_by_forum.get(filters['forum'].lower(), []):
                candidate_ids[cid] += 1
        if 'client_name' in filters:
            for cid in meta_by_client.get(filters['client_name'].lower(), []):
                candidate_ids[cid] += 1
        if 'matter_id' in filters:
            for cid in meta_by_matter.get(filters['matter_id'].lower(), []):
                candidate_ids[cid] += 1

        # Also check tags in the query
        q_lower = query.raw_query.lower()
        for tag_key in meta_by_tag:
            if tag_key in q_lower:
                for cid in meta_by_tag[tag_key]:
                    candidate_ids[cid] += 1

        if not candidate_ids:
            return []

        # Sort by match count (more filter dimensions matched = higher score)
        sorted_ids = sorted(candidate_ids.items(), key=lambda x: x[1], reverse=True)
        results = []
        for cid, match_count in sorted_ids[:limit]:
            chunk = chunk_id_map.get(cid)
            if chunk:
                score = float(match_count)  # Simple count-based score
                results.append(_chunk_to_candidate(chunk, 'metadata', score))
        return results


# ── Instantiate retrievers ───────────────────────────────────────────────────
RETRIEVERS: dict[str, object] = {
    'bm25': BM25Retriever(),
    'vector': VectorRetriever(),
    'metadata': MetadataRetriever(),
}

print('✅ Retrieval channels ready: bm25, vector, metadata')

## 10. Hybrid Fusion + Reranking Pipeline

Full retrieval pipeline with:
1. **Parallel channel execution** (ThreadPoolExecutor)
2. **Candidate dedup + provenance merge**
3. **RRF Fusion** (k=60)
4. **Cross-encoder reranking** with blended scoring

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _merge_candidates(channel_results: dict[str, list[Candidate]]) -> list[Candidate]:
    """Dedup candidates across channels. Merge provenance for multi-channel hits."""
    pool: dict[str, Candidate] = {}

    for channel, candidates in channel_results.items():
        for c in candidates:
            if c.chunk_id in pool:
                # Merge provenance — candidate found by multiple channels
                existing = pool[c.chunk_id]
                existing.provenance[channel] = c.raw_score
            else:
                pool[c.chunk_id] = c

    return list(pool.values())


def rrf_fuse(channel_results: dict[str, list[Candidate]], limit: int = 60) -> list[Candidate]:
    """Reciprocal Rank Fusion across channels with provenance."""
    scores: dict[str, float] = defaultdict(float)
    payload: dict[str, Candidate] = {}
    k = 60  # RRF constant

    for channel, candidates in channel_results.items():
        for rank, c in enumerate(candidates, start=1):
            scores[c.chunk_id] += 1.0 / (k + rank)
            if c.chunk_id not in payload:
                payload[c.chunk_id] = c
            else:
                # Merge provenance
                payload[c.chunk_id].provenance[channel] = c.raw_score

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    out = []
    for cid, score in ranked[:limit]:
        candidate = payload[cid]
        candidate.fused_score = score
        out.append(candidate)
    return out


def _minmax(values: list[float]) -> list[float]:
    if not values:
        return []
    lo, hi = min(values), max(values)
    if hi - lo < 1e-9:
        return [0.5] * len(values)
    return [(v - lo) / (hi - lo) for v in values]


def rerank_candidates(query: str, candidates: list[Candidate]) -> list[Candidate]:
    """Cross-encoder reranking with blended score (CE + RRF)."""
    if len(candidates) <= 1:
        return candidates

    pairs = [(query, c.text[:1200]) for c in candidates]
    ce_raw = [float(s) for s in reranker.predict(pairs, batch_size=32, show_progress_bar=False)]
    fused_raw = [c.fused_score for c in candidates]
    ce_n = _minmax(ce_raw)
    fu_n = _minmax(fused_raw)
    alpha = config.rerank_ce_weight

    for c, ce, fu, raw in zip(candidates, ce_n, fu_n, ce_raw):
        c.ce_score = raw
        c.rerank_score = alpha * ce + (1.0 - alpha) * fu

    candidates.sort(key=lambda c: c.rerank_score, reverse=True)
    return candidates


# ── Main Retrieval Pipeline ──────────────────────────────────────────────────

@dataclass
class RetrievalTrace:
    """Full observability trace for a single retrieval request."""
    query: str
    intent: str
    entities: dict
    channels_planned: list[str]
    channel_counts: dict[str, int] = field(default_factory=dict)
    unique_candidates: int = 0
    fused_count: int = 0
    final_count: int = 0
    latency: dict[str, float] = field(default_factory=dict)
    # Per-result provenance
    results: list[dict] = field(default_factory=list)


def retrieve(query: str, k: int | None = None) -> tuple[list[Candidate], RetrievalTrace]:
    """Full hybrid retrieval pipeline with query understanding + planner."""
    k = k or config.retrieve_k
    limit = config.retrieve_limit

    t_total = time.perf_counter()

    # ── Query Understanding ──────────────────────────────────────────────
    t0 = time.perf_counter()
    pq = understand_query(query)
    channels_to_run = plan_retrieval(pq)
    understanding_ms = round((time.perf_counter() - t0) * 1000, 1)

    trace = RetrievalTrace(
        query=query,
        intent=pq.intent.name,
        entities=pq.entities,
        channels_planned=channels_to_run,
    )
    trace.latency['understanding_ms'] = understanding_ms

    # ── Parallel Channel Execution ───────────────────────────────────────
    channel_results: dict[str, list[Candidate]] = {}

    with ThreadPoolExecutor(max_workers=len(channels_to_run)) as pool:
        t0 = time.perf_counter()
        futures = {}
        for ch_name in channels_to_run:
            retriever = RETRIEVERS.get(ch_name)
            if retriever:
                futures[pool.submit(retriever.retrieve, pq, limit)] = ch_name

        for future in as_completed(futures):
            ch_name = futures[future]
            try:
                results = future.result()
                channel_results[ch_name] = results
                trace.channel_counts[ch_name] = len(results)
                trace.latency[f'{ch_name}_ms'] = round((time.perf_counter() - t0) * 1000, 1)
            except Exception as e:
                print(f'  ⚠️ Channel {ch_name} failed: {e}')
                channel_results[ch_name] = []
                trace.channel_counts[ch_name] = 0

    trace.latency['parallel_search_ms'] = round((time.perf_counter() - t_total) * 1000, 1)

    # Count unique candidates before fusion
    all_ids = set()
    for results in channel_results.values():
        for c in results:
            all_ids.add(c.chunk_id)
    trace.unique_candidates = len(all_ids)

    # ── RRF Fusion ───────────────────────────────────────────────────────
    t1 = time.perf_counter()
    fused = rrf_fuse(channel_results, limit=limit)
    trace.fused_count = len(fused)
    trace.latency['fusion_ms'] = round((time.perf_counter() - t1) * 1000, 1)

    # ── Cross-Encoder Reranking ──────────────────────────────────────────
    t2 = time.perf_counter()
    reranked = rerank_candidates(query, fused)
    trace.latency['rerank_ms'] = round((time.perf_counter() - t2) * 1000, 1)

    final = reranked[:k]
    trace.final_count = len(final)
    trace.latency['total_ms'] = round((time.perf_counter() - t_total) * 1000, 1)

    # Build per-result provenance for the debugger
    trace.results = [
        {
            'rank': i + 1,
            'chunk_id': c.chunk_id,
            'filename': c.filename,
            'page': c.page_number,
            'channels': list(c.provenance.keys()),
            'channel_scores': c.provenance,
            'fused_score': round(c.fused_score, 4),
            'ce_score': round(c.ce_score, 4),
            'rerank_score': round(c.rerank_score, 4),
            'snippet': c.text[:120],
        }
        for i, c in enumerate(final)
    ]

    return final, trace


print('✅ Retrieval pipeline ready (with planner + provenance + trace)')

## 11. LLM Answer Generation

Same structured JSON output as the production system. Fallback chain: Groq → Gemini → extractive.

In [ ]:
import json

SYSTEM_PROMPT = """You are a legal DMS (Document Management System) assistant for a law firm. Answer ONLY from the provided document excerpts below.

Rules:
- Cite every claim inline as [filename, p.N] where N is the page number.
- If the excerpts do not contain enough information, set "abstain" to true and leave "answer" empty.
- Never invent facts not found in the excerpts.
- Write in clear, professional prose like a research memo.
- Identify the single most relevant document as the primary_document.
- List additional relevant documents as supporting_documents.
- Produce a "key_finding" — a 1-2 sentence executive summary of the main takeaway.
- Include matter_id, document_type, and tags from the excerpt headers when available.

Respond with VALID JSON only — no markdown fences:
{
  "abstain": false,
  "key_finding": "One-line summary of the key finding across all documents.",
  "answer": "Detailed answer with inline citations like [Affidavit - CA 10046 of 2025.pdf, p.3]",
  "primary_document": {
    "filename": "exact filename from excerpt header",
    "page": 3,
    "matter_id": "if available from excerpt header, else null",
    "document_type": "if available from excerpt header, else null",
    "tags": ["tag1", "tag2"],
    "snippet": "verbatim short excerpt (<=120 chars)"
  },
  "supporting_documents": [
    {
      "filename": "another_file.pdf",
      "page": 1,
      "matter_id": "if available",
      "document_type": "if available",
      "tags": ["tag1"],
      "snippet": "verbatim short excerpt (<=120 chars)"
    }
  ],
  "citations": [
    {"file": "exact filename", "page": 3, "snippet": "verbatim short excerpt (<=120 chars)"}
  ]
}"""


def _pack_context(hits: list[Candidate]) -> str:
    """Pack retrieval hits into numbered context for the LLM."""
    parts = []
    for i, h in enumerate(hits, start=1):
        header = f"[{i}] {h.filename} | page {h.page_number}"
        meta_parts = []
        if h.document_type:
            meta_parts.append(f"Type: {h.document_type}")
        if h.matter_id:
            meta_parts.append(f"Matter: {h.matter_id}")
        if h.tags:
            meta_parts.append(f"Tags: {', '.join(h.tags)}")
        if meta_parts:
            header += f" | {' | '.join(meta_parts)}"
        parts.append(f"{header}\n{h.text[:1200]}")
    return '\n\n---\n\n'.join(parts)


def _parse_raw(raw: str) -> dict | None:
    raw = re.sub(r'^```(?:json)?\s*', '', (raw or '').strip())
    raw = re.sub(r'\s*```$', '', raw)
    try:
        data = json.loads(raw)
        return {
            'abstain': bool(data.get('abstain', False)),
            'key_finding': str(data.get('key_finding') or ''),
            'answer': str(data.get('answer') or ''),
            'primary_document': data.get('primary_document'),
            'supporting_documents': data.get('supporting_documents') or [],
            'citations': data.get('citations') or [],
        }
    except (json.JSONDecodeError, TypeError, ValueError):
        return None


def _extractive_answer(hits: list[Candidate]) -> dict:
    snippets = [
        f"[{h.filename}, p.{h.page_number}] {h.text[:400]}"
        for h in hits[:3]
    ]
    primary = None
    supporting = []
    for i, h in enumerate(hits[:5]):
        doc_entry = {
            'filename': h.filename,
            'page': h.page_number,
            'matter_id': h.matter_id,
            'document_type': h.document_type,
            'tags': h.tags or [],
            'snippet': h.text[:120],
        }
        if i == 0:
            primary = doc_entry
        else:
            supporting.append(doc_entry)

    return {
        'abstain': False,
        'key_finding': f'Found {len(hits)} relevant excerpts from firm documents.',
        'answer': 'Based on the documents:\n\n' + '\n\n'.join(snippets) if snippets else 'No relevant documents found.',
        'primary_document': primary,
        'supporting_documents': supporting,
        'citations': [
            {'file': h.filename, 'page': h.page_number, 'snippet': h.text[:120]}
            for h in hits[:3]
        ],
    }


def _call_groq(query: str, context: str) -> str:
    from groq import Groq
    client = Groq(api_key=config.groq_api_key)
    r = client.chat.completions.create(
        model=config.groq_model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Question: {query}\n\nExcerpts:\n{context}'},
        ],
        max_tokens=4096,
        temperature=0.1,
        response_format={'type': 'json_object'},
        timeout=30,
    )
    msg = r.choices[0].message
    return msg.content or getattr(msg, 'reasoning_content', None) or ''


def _call_gemini(query: str, context: str) -> str:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=config.gemini_api_key)
    prompt = f'{SYSTEM_PROMPT}\n\nQuestion: {query}\n\nExcerpts:\n{context}'
    r = client.models.generate_content(
        model=config.gemini_model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            max_output_tokens=4096,
            temperature=0.1,
        ),
    )
    return r.text or ''


def complete(query: str, hits: list[Candidate]) -> dict:
    """Try Groq -> Gemini -> extractive fallback."""
    if not hits:
        return {**_extractive_answer([]), 'provider': 'extractive'}

    context = _pack_context(hits)
    errors: list[str] = []

    providers = [
        ('groq', config.groq_api_key, _call_groq),
        ('gemini', config.gemini_api_key, _call_gemini),
    ]

    for name, api_key, call_fn in providers:
        if not api_key:
            continue
        try:
            raw = call_fn(query, context)
            result = _parse_raw(raw)
            if result is not None:
                result['provider'] = name
                return result
            errors.append(f'{name}: parse error on {raw[:80]!r}')
        except Exception as exc:
            errors.append(f'{name}: {type(exc).__name__}: {str(exc)[:120]}')

    result = _extractive_answer(hits)
    result['provider'] = 'extractive'
    if errors:
        result['llm_errors'] = errors
    return result


print('✅ LLM answer generation ready')

## 12. Mount Google Drive (Optional)

If your PDFs are on Google Drive, run this cell to mount it.

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# print('Google Drive mounted at /content/drive')

## 13. Ingest Documents

Set `FOLDER_PATH` to the folder containing your PDFs and run the cell.

You can also ingest via the **Gradio UI** (next cell) — the "📂 Ingest" tab.

In [ ]:
# ── Set your folder path here ────────────────────────────────────────────────
# Examples:
#   Local:         '/content/my_pdfs'
#   Google Drive:  '/content/drive/MyDrive/Legal_Documents'
#   Upload:        Upload files to /content/ first, then use '/content/'

FOLDER_PATH = '/content/drive/MyDrive/Legal_Documents'  # <- Change this!

status = ingest_folder(FOLDER_PATH)

## 14. Quick Test — Retrieval Pipeline with Debug Output

In [ ]:
test_query = 'What are the force majeure claims in CERC petitions?'

hits, trace = retrieve(test_query)

print(f'Query:   "{trace.query}"')
print(f'Intent:  {trace.intent}')
print(f'Entities: {trace.entities}')
print(f'Channels: {trace.channels_planned}')
print()

# Channel breakdown
for ch, count in trace.channel_counts.items():
    ms = trace.latency.get(f'{ch}_ms', 0)
    print(f'  {ch:12s} → {count:3d} candidates ({ms:.0f}ms)')

print(f'\n  Unique candidates: {trace.unique_candidates}')
print(f'  After fusion:      {trace.fused_count}')
print(f'  After rerank:      {trace.final_count}')
print(f'  Total:             {trace.latency["total_ms"]:.0f}ms')

print(f'\n--- Top {len(hits)} Results ---')
for r in trace.results[:5]:
    channels = ', '.join(r['channels'])
    print(f"  [{r['rank']}] {r['filename']} p.{r['page']}")
    print(f"      Channels: {channels}")
    print(f"      Scores: fused={r['fused_score']:.4f} ce={r['ce_score']:.4f} rerank={r['rerank_score']:.4f}")
    print(f"      {r['snippet'][:80]}...")
    print()

# LLM answer
result = complete(test_query, hits)
print(f'\n--- Answer (via {result.get("provider", "?")}) ---')
print(f'Key Finding: {result.get("key_finding", "")}')
print(f'\n{result.get("answer", "")[:500]}')

## 15. 🚀 Launch Interactive Q&A Interface (Gradio)

Three tabs:
1. **💬 Ask** — type questions continuously, get structured answers
2. **🔍 Debug** — retrieval debugger showing channel breakdown, provenance, scores
3. **📂 Ingest** — point to a new folder anytime to re-ingest

Launches with a **public URL** (`share=True`) — stays active as long as the Colab session runs.

In [ ]:
import gradio as gr


def format_answer(result: dict, trace: RetrievalTrace, hits: list[Candidate]) -> str:
    """Format the LLM result into readable markdown."""
    parts = []

    kf = result.get('key_finding', '')
    if kf:
        parts.append(f'### 🔑 Key Finding\n{kf}')

    answer = result.get('answer', '')
    if answer:
        parts.append(f'### 📝 Answer\n{answer}')

    pd = result.get('primary_document')
    if pd:
        tags_str = ', '.join(pd.get('tags', [])) if pd.get('tags') else 'None'
        parts.append(
            f'### 📄 Primary Document\n'
            f'- **File:** {pd.get("filename", "?")}\n'
            f'- **Page:** {pd.get("page", "?")}\n'
            f'- **Type:** {pd.get("document_type", "?")}\n'
            f'- **Matter ID:** {pd.get("matter_id", "N/A")}\n'
            f'- **Tags:** {tags_str}\n'
            f'- **Snippet:** {pd.get("snippet", "")}'
        )

    sd = result.get('supporting_documents', [])
    if sd:
        sd_lines = ['### 📚 Supporting Documents']
        for i, doc in enumerate(sd, 1):
            tags_str = ', '.join(doc.get('tags', [])) if doc.get('tags') else 'None'
            sd_lines.append(
                f'{i}. **{doc.get("filename", "?")}** (p.{doc.get("page", "?")}) '
                f'— {doc.get("document_type", "?")} | Tags: {tags_str}'
            )
        parts.append('\n'.join(sd_lines))

    # Latency + metadata
    parts.append(
        f'---\n'
        f'⚡ **Latency:** {trace.latency.get("total_ms", 0):.0f}ms total '
        f'(search={trace.latency.get("parallel_search_ms", 0):.0f}ms, '
        f'rerank={trace.latency.get("rerank_ms", 0):.0f}ms)  \n'
        f'🧠 **Intent:** {trace.intent} | '
        f'**Channels:** {", ".join(trace.channels_planned)} | '
        f'**Provider:** {result.get("provider", "?")} | '
        f'**Hits:** {trace.final_count}'
    )

    if result.get('llm_errors'):
        parts.append(f'⚠️ **LLM Errors:** {" | ".join(result["llm_errors"])}')

    return '\n\n'.join(parts)


def format_debug(trace: RetrievalTrace) -> str:
    """Format the retrieval trace into a debugger view."""
    lines = []
    lines.append(f'## 🔍 Retrieval Debug: "{trace.query}"')
    lines.append(f'\n**Intent:** `{trace.intent}`')
    if trace.entities:
        ent_str = ', '.join(f'{k}={v}' for k, v in trace.entities.items())
        lines.append(f'**Entities:** {ent_str}')
    lines.append(f'**Channels planned:** `{", ".join(trace.channels_planned)}`')

    # Channel breakdown
    lines.append('\n### Channel Results')
    lines.append('| Channel | Candidates | Latency |')
    lines.append('|---------|-----------|---------|')
    for ch in trace.channels_planned:
        count = trace.channel_counts.get(ch, 0)
        ms = trace.latency.get(f'{ch}_ms', 0)
        lines.append(f'| {ch} | {count} | {ms:.0f}ms |')

    # Pipeline summary
    lines.append('\n### Pipeline Summary')
    lines.append(f'```')
    total_raw = sum(trace.channel_counts.values())
    lines.append(f'Raw candidates:    {total_raw}')
    lines.append(f'Unique candidates: {trace.unique_candidates}')
    lines.append(f'After fusion:      {trace.fused_count}')
    lines.append(f'After rerank:      {trace.final_count}')
    lines.append(f'```')

    # Per-result provenance
    if trace.results:
        lines.append('\n### Result Provenance')
        lines.append('| Rank | File | Page | Channels | Fused | CE | Rerank |')
        lines.append('|------|------|------|----------|-------|-----|--------|')
        for r in trace.results:
            channels = ', '.join(r['channels'])
            ch_check = ''
            for ch in trace.channels_planned:
                ch_check += f' {"✓" if ch in r["channels"] else "✗"}{ch}'
            lines.append(
                f'| {r["rank"]} | {r["filename"][:30]} | {r["page"]} | '
                f'{ch_check.strip()} | {r["fused_score"]:.4f} | '
                f'{r["ce_score"]:.4f} | {r["rerank_score"]:.4f} |'
            )

    # Latency breakdown
    lines.append('\n### Latency Breakdown')
    lines.append('| Stage | Time |')
    lines.append('|-------|------|')
    for stage, ms in trace.latency.items():
        lines.append(f'| {stage} | {ms:.1f}ms |')

    return '\n'.join(lines)


def ask_question(query: str, num_results: int = 8) -> str:
    if not query.strip():
        return '⚠️ Please enter a question.'
    if not all_chunks:
        return '❌ No documents ingested yet. Use the **📂 Ingest** tab first.'
    hits, trace = retrieve(query, k=int(num_results))
    result = complete(query, hits)
    return format_answer(result, trace, hits)


def debug_question(query: str, num_results: int = 8) -> str:
    if not query.strip():
        return '⚠️ Please enter a question.'
    if not all_chunks:
        return '❌ No documents ingested yet. Use the **📂 Ingest** tab first.'
    _, trace = retrieve(query, k=int(num_results))
    return format_debug(trace)


def ingest_and_report(folder_path: str) -> str:
    if not folder_path.strip():
        return '⚠️ Please enter a folder path.'
    return ingest_folder(folder_path.strip())


# ── Build Gradio Interface ────────────────────────────────────────────────────

with gr.Blocks(
    title='Legal DMS — RAG Q&A',
    theme=gr.themes.Soft(primary_hue='blue', secondary_hue='slate'),
) as demo:
    gr.Markdown(
        '# 📜 Legal DMS — Production-Grade Document Q&A\n'
        'Hybrid retrieval with **query understanding → retrieval planner → '
        'parallel channels (BM25 + vector + metadata) → RRF fusion → cross-encoder rerank → LLM**.'
    )

    # ── Tab 1: Ask ────────────────────────────────────────────────────────
    with gr.Tab('💬 Ask'):
        with gr.Row():
            query_input = gr.Textbox(
                label='Your Question',
                placeholder='e.g., What are the force majeure claims in the CERC petitions?',
                lines=2, scale=4,
            )
            num_results = gr.Slider(
                minimum=3, maximum=20, value=8, step=1,
                label='Results (k)', scale=1,
            )
        ask_btn = gr.Button('🔍 Search & Answer', variant='primary', size='lg')
        answer_output = gr.Markdown(label='Answer')
        ask_btn.click(fn=ask_question, inputs=[query_input, num_results], outputs=answer_output)
        query_input.submit(fn=ask_question, inputs=[query_input, num_results], outputs=answer_output)

    # ── Tab 2: Debug ──────────────────────────────────────────────────────
    with gr.Tab('🔍 Debug'):
        gr.Markdown(
            '### Retrieval Debugger\n'
            'Shows the full retrieval trace: query understanding, channel breakdown, '
            'candidate provenance, and per-stage latency. No LLM call.'
        )
        with gr.Row():
            dbg_query = gr.Textbox(
                label='Query to Debug',
                placeholder='e.g., Find all CERC affidavits about tariff disputes',
                lines=2, scale=4,
            )
            dbg_k = gr.Slider(
                minimum=3, maximum=20, value=8, step=1,
                label='Results (k)', scale=1,
            )
        dbg_btn = gr.Button('🔬 Debug Retrieval', variant='secondary', size='lg')
        dbg_output = gr.Markdown(label='Debug Trace')
        dbg_btn.click(fn=debug_question, inputs=[dbg_query, dbg_k], outputs=dbg_output)
        dbg_query.submit(fn=debug_question, inputs=[dbg_query, dbg_k], outputs=dbg_output)

    # ── Tab 3: Ingest ─────────────────────────────────────────────────────
    with gr.Tab('📂 Ingest'):
        gr.Markdown(
            'Point to a folder containing PDF files. All PDFs will be:\n'
            '1. Text-extracted (pypdf)\n'
            '2. Chunked (1200 chars, 150 overlap)\n'
            '3. Metadata-extracted (document type, forum, case number, tags)\n'
            '4. Embedded (MiniLM-L6) and indexed (ChromaDB + BM25 + metadata indices)'
        )
        folder_input = gr.Textbox(
            label='Folder Path',
            placeholder='/content/drive/MyDrive/Legal_Documents',
            lines=1,
        )
        ingest_btn = gr.Button('📥 Ingest PDFs', variant='primary', size='lg')
        ingest_output = gr.Textbox(label='Ingestion Status', lines=15, interactive=False)
        ingest_btn.click(fn=ingest_and_report, inputs=folder_input, outputs=ingest_output)

    # ── Tab 4: About ──────────────────────────────────────────────────────
    with gr.Tab('ℹ️ Architecture'):
        gr.Markdown(
            '## Pipeline Architecture\n'
            '```\n'
            '                    QUERY\n'
            '                      │\n'
            '             Query Understanding\n'
            '              (Intent + Entities)\n'
            '                      │\n'
            '             Retrieval Planner\n'
            '              (Adaptive Channels)\n'
            '                      │\n'
            '        ┌─────────────┼─────────────┐\n'
            '        ▼             ▼             ▼\n'
            '   BM25 Keyword  Vector Cosine  Metadata\n'
            '        │             │             │\n'
            '        └─────────────┼─────────────┘\n'
            '                      ▼\n'
            '             Candidate Pool + Dedup\n'
            '               (with Provenance)\n'
            '                      │\n'
            '                RRF Fusion\n'
            '                      │\n'
            '              Cross-Encoder Rerank\n'
            '                      │\n'
            '                LLM Answer + Citations\n'
            '```\n\n'
            '### Query Intent → Channel Plan\n'
            '| Intent | Channels |\n'
            '|--------|----------|\n'
            '| EXACT_LOOKUP | metadata, bm25 |\n'
            '| NORMAL_RESEARCH | bm25, vector, metadata |\n'
            '| SIMILAR_MATTER | vector, metadata |\n'
            '| CLIENT_HISTORY | metadata, bm25 |\n'
            '| GENERAL | bm25, vector |\n\n'
            '### Models\n'
            f'- **Embedder:** `{config.embedding_model}`\n'
            f'- **Reranker:** `{config.rerank_model}`\n'
            f'- **LLM:** Groq (`{config.groq_model}`) → Gemini (`{config.gemini_model}`) → Extractive\n\n'
            '### Settings\n'
            f'- Chunk size: {config.chunk_max_chars} chars, overlap: {config.chunk_overlap}\n'
            f'- Retrieve top-{config.retrieve_limit}, return top-{config.retrieve_k}\n'
            f'- Rerank CE weight (α): {config.rerank_ce_weight}\n\n'
            '### Storage (Colab Edition)\n'
            '| Abstraction | Implementation |\n'
            '|-------------|---------------|\n'
            '| VectorStore | ChromaDB (in-memory) |\n'
            '| SearchStore | BM25Okapi (in-memory) |\n'
            '| MetadataStore | Python dicts (in-memory) |\n\n'
            '### Production Path\n'
            '| Abstraction | Production |\n'
            '|-------------|------------|\n'
            '| VectorStore | pgvector / Qdrant |\n'
            '| SearchStore | Postgres FTS / Elasticsearch |\n'
            '| MetadataStore | Postgres tables |\n'
            '| GraphStore | Postgres → Neo4j |\n'
        )


# Launch with public URL for continuous Q&A
demo.launch(share=True, show_error=True)